In [ ]:
from datasets import load_dataset

e:\projects\Math Reasoning RL\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
ds = load_dataset("openai/gsm8k", "main")

In [3]:
ds

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 7473
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 1319
    })
})

In [4]:
ds.column_names

{'train': ['question', 'answer'], 'test': ['question', 'answer']}

In [5]:
ds['train'].features

{'question': Value('string'), 'answer': Value('string')}

In [6]:
ds['train'][2]

{'question': 'Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?',
 'answer': "In the beginning, Betty has only 100 / 2 = $<<100/2=50>>50.\nBetty's grandparents gave her 15 * 2 = $<<15*2=30>>30.\nThis means, Betty needs 100 - 50 - 30 - 15 = $<<100-50-30-15=5>>5 more.\n#### 5"}

In [13]:
def extract_thinking(data):
    answer_parts = data['answer'].split("####")
    answer = answer_parts[1].strip().replace(",","")
    thinking = answer_parts[0].strip()
    return {'question': data['question'],'thinking':thinking,'answer':answer}

ds = ds.map(extract_thinking)

Map: 100%|██████████| 1319/1319 [00:00<00:00, 20469.18 examples/s]


In [15]:
ds['test'][2]

{'question': 'Josh decides to try flipping a house.  He buys a house for $80,000 and then puts in $50,000 in repairs.  This increased the value of the house by 150%.  How much profit did he make?',
 'answer': '70000',
 'thinking': 'The cost of the house and repairs came out to 80,000+50,000=$<<80000+50000=130000>>130,000\nHe increased the value of the house by 80,000*1.5=<<80000*1.5=120000>>120,000\nSo the new value of the house is 120,000+80,000=$<<120000+80000=200000>>200,000\nSo he made a profit of 200,000-130,000=$<<200000-130000=70000>>70,000'}

In [16]:
system_prompt = (
    "Solve the math problem below.\n"
    "First, reason step by step inside <think> </think> tags.\n"
    "Then give the final answer inside <answer> </answer> tags.\n"
    "The answer must contain only the final number, with no units, symbols, or commas.\n"
    "Format:\n"
    "<think>\n...\n</think>\n"
    "<answer>\n...\n</answer>"
)
def create_prompt(data):
    return {"prompt":[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": data['question']}, 
    ]}

clean_ds = ds.map(create_prompt)

Map: 100%|██████████| 1319/1319 [00:00<00:00, 14200.43 examples/s]


In [17]:
clean_ds['train'][0]

{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?',
 'answer': '72',
 'thinking': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.',
 'prompt': [{'role': 'system',
   'content': 'Solve the math problem below.\nFirst, reason step by step inside <think> </think> tags.\nThen give the final answer inside <answer> </answer> tags.\nThe answer must contain only the final number, with no units, symbols, or commas.\nFormat:\n<think>\n...\n</think>\n<answer>\n...\n</answer>'},
  {'role': 'user',
   'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?'}]}

In [ ]:
import re

def extract_number_and_unit(text):
    pattern = r"([$€£¥]?)\s*([-+]?\d[\d,]*\.?\d*)\s*([a-zA-Z°%/]*)"
    match = re.search(pattern, text)
    
    if not match:
        return [None, None]
        
    prefix_unit, number_str, suffix_unit = match.groups()
    number = float(number_str.replace(",", ""))
    unit = prefix_unit or suffix_unit
    return [number, unit]

def is_formatted(gen_answer):
    if re.fullmatch(r"\s*<think>.*?</think>\s*<answer>.*?</answer>\s*", gen_answer, re.DOTALL):
        return True
    return False

def is_correct(gen_answer, actual_answer):
    match = re.search(r"<answer>(.*?)</answer>", gen_answer, re.DOTALL)
    ans = extract_number_and_unit(match.group(1).strip())[0] if match else None
    if(ans is not None and abs(float(ans) - float(actual_answer)) < 1e-6):
        return True
    return False

#Reward Function
def _reward(gen_answer, actual_answer):
    reward = 0.
    #Format Reward
    if is_formatted(gen_answer):
        reward += .5
    else:
        reward -= .5

    #Compare Answers Reward
    if is_correct(gen_answer, actual_answer):
        reward += 1
    else:
        reward -= 1

    return reward

def batch_reward(completions, answer, **kwargs):
    reward_b = []
    for i in range(len(completions)):
        reward_b.append(_reward(completions[i][0]["content"], answer[i]))
    return reward_b

In [ ]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    #Training Hyperparameters
    output_dir="outputs/qwen3-grpo",
    num_generations=8,
    temperature=1.,
    max_completion_length=512,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=1e-6,
    max_steps=100,

    #Optimization
    bf16=True,
    gradient_checkpointing=True,

    #Logging
    logging_steps=1,
    report_to="tensorboard",
    run_name="qwen3-1.7b-gsm8k-try1",
    log_completions=True, 

    #Saving
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,

    #Validation
    eval_strategy="steps",
    eval_steps=50,
    per_device_eval_batch_size=8,
    
)

trainer = GRPOTrainer(
    model="Qwen/Qwen3-1.7B",
    reward_funcs=[batch_reward],
    args=training_args,
    train_dataset=clean_ds["train"],
    eval_dataset=clean_ds['test'].select(range(200))
)

trainer.train()
trainer.save_model("outputs/qwen3-grpo/final")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_path = "outputs/qwen3-grpo/final"

test_ds = clean_ds['test']

tokenizer = AutoTokenizer.from_pretrained(model_path, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(model_path, dtype=torch.bfloat16, device_map="cuda")
model.eval()

batch_size = 32

n_format = 0
n_correct = 0

for start in range(0,len(test_ds), batch_size):
    batch = test_ds[start : start + batch_size]

    text = [
        tokenizer.apply_chat_template(p, tokenize=False, add_generation_prompt=True)
        for p in batch["prompt"]
    ]

    inputs = tokenizer(text, return_tensors="pt", padding=True).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=512, do_sample=False)

    new_ids = output_ids[:, inputs["input_ids"].shape[1]:]
    completions = tokenizer.batch_decode(new_ids, skip_special_tokens=True)

    for out, gold in zip(completions, batch["answer"]):
        n_format += is_formatted(out)
        n_correct += is_correct(out, gold)

    print(f"{start + len(completions)}/{len(test_ds)} done")


print(f"format rate: {n_format / len(test_ds):.1%}")
print(f"accuracy:    {n_correct / len(test_ds):.1%}")